# 01 — Prepare Held-Out Test Set

Creates a held-out test set for final model evaluation. This test set is **never used during training**.

**Data provenance:**
- **Positives** (label=1): ALL phage-host pairs from PBI-Scope's `phage_host_associations` table
  that are not marked as negative in `private_interactions`.
- **Negatives** (label=0): Two sources combined:
  1. `private_data`: Pairs with explicit negative interaction in `private_interactions`
  2. `generated`: Random pairs not in the positive set (capped at 2x private negatives)

**Output files** (saved in `test_data/`):
- `test_set.csv` — string IDs, labels, and sources (human-readable)
- `excluded_pairs.csv` — Phage_ID,Host_ID to pass to `train.py --exclude-ids`
- `test_set.npz` — precomputed numpy arrays for `02_evaluate_model.ipynb`

**Run this once** before training. The test set is fixed and reusable across experiments.

## 1. Connect to PBI-Scope

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd()))
from pbi import quick_connect
from pbi.negative_examples import NegativeExampleGenerator
from pbi_adapter import PBIAdapter

In [ ]:
retriever = quick_connect()
adapter = PBIAdapter(retriever)

BACTERIUM_THRESHOLD = 7_000_000
PHAGE_THRESHOLD = 200_000

## 2. Load All Pairs and Classify

Query all phage-host pair IDs from PBI-Scope (fast — no sequences fetched).
Classify each pair using the `private_interactions` table:
- Pairs with negative interaction type → negatives
- Everything else → positives

In [ ]:
all_pairs = adapter.get_pair_ids_only(shuffle=True)
print(f"Total pairs in database: {len(all_pairs):,}")

positive_pairs, private_negatives = adapter.classify_pairs_by_interaction(all_pairs)
print(f"\nPositive pairs: {len(positive_pairs):,}")
print(f"True negatives (private data): {len(private_negatives):,}")

## 3. Generate Synthetic Negatives

Generate random phage-host pairs that are NOT in the positive set.
Capped at 2x the number of private negatives to avoid overwhelming the dataset.

In [ ]:
# Calculate cap before generation to avoid wasted time
if len(private_negatives) > 0:
    max_generated = max(len(private_negatives) * 2, 100)
else:
    max_generated = len(positive_pairs)

target_count = min(len(positive_pairs), max_generated)
ratio = target_count / len(positive_pairs) if len(positive_pairs) > 0 else 0

print(f"Private negatives: {len(private_negatives):,}")
print(f"Generating {target_count:,} synthetic negatives (ratio={ratio:.3f})...")

neg_gen = NegativeExampleGenerator(retriever)
generated_negatives = neg_gen.generate_random_negatives(
    positive_pairs, ratio=ratio
)

# Deduplicate against private negatives
if len(private_negatives) > 0:
    private_neg_set = set(
        zip(private_negatives["Phage_ID"], private_negatives["Host_ID"])
    )
    before = len(generated_negatives)
    generated_negatives = generated_negatives[
        ~generated_negatives.apply(
            lambda r: (r["Phage_ID"], r["Host_ID"]) in private_neg_set, axis=1
        )
    ].reset_index(drop=True)
    deduped = before - len(generated_negatives)
    if deduped > 0:
        print(f"Removed {deduped} duplicates against private negatives")

generated_negatives["negative_source"] = "generated"
print(f"Generated negatives: {len(generated_negatives):,}")

## 4. Combine and Split

Merge positives and negatives into a single DataFrame, then split off 15% as the held-out test set.
The split is stratified by class and negative source.

In [ ]:
# Combine into single DataFrame with labels
pos_df = positive_pairs.copy()
pos_df["label"] = 1
pos_df["source"] = "positive"

priv_neg_df = private_negatives.copy()
priv_neg_df["label"] = 0
priv_neg_df["source"] = "private_data"

gen_neg_df = generated_negatives[["Phage_ID", "Host_ID"]].copy()
gen_neg_df["label"] = 0
gen_neg_df["source"] = "generated"

all_data = pd.concat([pos_df, priv_neg_df, gen_neg_df], ignore_index=True)
print(f"Total pairs: {len(all_data):,}")
print(f"  Positives: {int(all_data['label'].sum()):,}")
print(f"  Negatives: {int(len(all_data) - all_data['label'].sum()):,}")
print(f"\nSource breakdown:")
print(all_data["source"].value_counts().to_string())

In [ ]:
# Stratified split: 85% train, 15% test
stratify_key = all_data.apply(
    lambda r: f"pos" if r["label"] == 1 else f"neg_{r['source']}", axis=1
)

train_df, test_df = train_test_split(
    all_data, stratify=stratify_key, test_size=0.15, shuffle=True, random_state=42
)

print(f"Train: {len(train_df):,} pairs")
print(f"Test:  {len(test_df):,} pairs")
print(f"\nTest set source breakdown:")
print(test_df["source"].value_counts().to_string())

## 5. Fetch and Encode Test Sequences

In [ ]:
print(f"Fetching sequences for {len(test_df)} test pairs...")

bacteria_seqs = []
phage_seqs = []
valid_mask = []

for i, row in test_df.iterrows():
    bseq = adapter._fetch_host_sequence(row["Host_ID"])
    pseq = adapter._fetch_phage_sequence(row["Phage_ID"])
    if bseq is not None and pseq is not None:
        bacteria_seqs.append(adapter._pad_and_encode(bseq, BACTERIUM_THRESHOLD))
        phage_seqs.append(adapter._pad_and_encode(pseq, PHAGE_THRESHOLD))
        valid_mask.append(True)
    else:
        valid_mask.append(False)

valid_mask = np.array(valid_mask)
n_dropped = len(valid_mask) - valid_mask.sum()
if n_dropped > 0:
    print(f"Dropped {n_dropped} pairs (missing/too-short sequences)")

bacteria_arr = np.stack(bacteria_seqs)
phage_arr = np.stack(phage_seqs)
test_df = test_df[valid_mask].reset_index(drop=True)

print(f"Encoded: bacteria={bacteria_arr.shape}, phage={phage_arr.shape}")

## 6. Save Test Set

In [ ]:
out_dir = Path.cwd() / "test_data"
out_dir.mkdir(exist_ok=True)

# Save test set CSV
test_df.to_csv(out_dir / "test_set.csv", index=False)
print(f"Saved test_set.csv ({len(test_df)} rows)")

# Save excluded pair IDs (for train.py --exclude-ids to prevent data leakage)
test_df[["Phage_ID", "Host_ID"]].to_csv(out_dir / "excluded_pairs.csv", index=False)
print(f"Saved excluded_pairs.csv ({len(test_df)} rows)")

# Save encoded sequences (for 02_evaluate_model.ipynb — no DB needed)
np.savez(
    out_dir / "test_set.npz",
    bacteria=bacteria_arr,
    phage=phage_arr,
    labels=test_df["label"].values.astype(np.float32),
    sources=test_df["source"].values,
)
print(f"Saved test_set.npz")

## Done

The held-out test set is ready. Files saved in `test_data/`:

| File | Contents |
|------|----------|
| `test_set.csv` | Phage_ID, Host_ID, label, source |
| `excluded_pairs.csv` | Phage_ID, Host_ID (for `--exclude-ids` in train.py) |
| `test_set.npz` | `bacteria` (N,T,4), `phage` (N,T,4), `labels` (N,), `sources` (N,) |

**Next steps:**
1. Train a model (excluding test pairs):
   ```bash
   python train.py --config config.yaml --exclude-ids test_data/excluded_pairs.csv
   ```
2. Evaluate: open `02_evaluate_model.ipynb`